<a href="https://colab.research.google.com/github/SANGHATI23/neuroimaging-fhir-omop-fidelity/blob/main/Phase47.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ICHI 2027 · Phase 47 — frozen TFL v15 runner audit and neuroimaging applicability preregistration

Phase 46 established the exact frozen generic TFL v15 comparator release but did not execute Arm C on the neuroimaging study.

Phase 47 answers two narrower questions before any comparator claim is made:

1. **Is the authoritative v15 runner itself reproducibly executable without modifying its method code?**
2. **Which neuroimaging challenge families are even in scope for the documented frozen generic warning vocabulary?**

The notebook resolves the exact Phase 46 commit, verifies the frozen archive hash, inspects the authoritative v15 integration notebook, audits its dependencies and external-data assumptions, and attempts an unchanged execution only when it is non-interactive and all required local assets appear present.

It also freezes a **documentation-derived applicability matrix** for E0–E7. That matrix is a preregistered expectation, not an Arm C result.

### Comparator boundary

The frozen v15 warning vocabulary is kept intact:

- `W_DEMOGRAPHIC_MISSING`
- `W_DUPLICATE_SOURCE_ID`
- `W_TRACEABILITY_LOSS`
- `W_CONFLICTING_CODING`
- `W_UNMAPPED_RESOURCE`
- `W_MEDICATION_ATTRIBUTION`
- `W_UNRESOLVED_REFERENCE`

No neuroimaging-specific laterality, unit, longitudinal-pair, model-version, TRACE-state, or reliability rule is added to Arm C.


In [1]:
# CODE CELL 1/9 — Drive, constants, helpers
import os, re, csv, json, hashlib, shutil, tempfile, zipfile, time, subprocess, sys, ast, importlib.util
import datetime as dt
from pathlib import Path
from collections import Counter, defaultdict

PROJECT='ICHI2027_neuroimaging_fhir_omop_fidelity'
PHASE='47_frozen_tfl_v15_runner_audit_and_neuroimaging_applicability_preregistration'

EXPECTED_P46_SHAREABLE='123aed1bed47a1b84660d9fad7dfb247b9cba2638dfb1f3cb8883169e7836d1e'
EXPECTED_P46_LOCK='d780c334823467cbe243cf623db3cc13be6e89e2ba33efd9c24d199f0299ab02'
EXPECTED_P41_CHALLENGE='6fdb81ebb3e7e7446b36fc2ea27534217560e88cc0e22f5f213f42868cb10a69'

REPO='SANGHATI23/ohdsi-fhir-omop-showcase-demo'
TAG='tfl-submission-freeze-v15'
COMMIT='a4b7265adabf096e84b957de068e24a5ab853425'
EXPECTED_COMMIT_ARCHIVE_SHA='e729610f050217a2c62c0536309c7e3ac3508f39992d7ed46b1b70c9bc9478dc'
RUNNER_NAME='FHIRy_pyOMOP_TFL_SubmissionFreezeRunner_v15.ipynb'

FROZEN_WARNINGS=[
 'W_DEMOGRAPHIC_MISSING','W_DUPLICATE_SOURCE_ID','W_TRACEABILITY_LOSS',
 'W_CONFLICTING_CODING','W_UNMAPPED_RESOURCE','W_MEDICATION_ATTRIBUTION',
 'W_UNRESOLVED_REFERENCE'
]
FROZEN_FATES=['PRESERVED','NORMALIZED','COLLAPSED','AMBIGUOUS','UNMAPPED','DROPPED','TRACEABILITY_LOSS']

def sha256(p):
    h=hashlib.sha256()
    with Path(p).open('rb') as f:
        for b in iter(lambda:f.read(2*1024*1024),b''):
            h.update(b)
    return h.hexdigest()

def write_csv(path,rows,fieldnames=None):
    rows=list(rows)
    if fieldnames is None:
        if not rows: raise ValueError('fieldnames required for empty CSV')
        fieldnames=list(rows[0].keys())
    with Path(path).open('w',newline='',encoding='utf-8') as f:
        w=csv.DictWriter(f,fieldnames=fieldnames)
        w.writeheader(); w.writerows(rows)

if os.environ.get('PHASE47_LOCAL_DRIVE_ROOT'):
    MYDRIVE=Path(os.environ['PHASE47_LOCAL_DRIVE_ROOT']).expanduser().resolve()
    ROOTS=[MYDRIVE]
    print('Offline Drive-mirror mode:',MYDRIVE)
else:
    from google.colab import drive
    drive.mount('/content/drive',force_remount=False)
    MYDRIVE=Path('/content/drive/MyDrive')
    assert MYDRIVE.is_dir()
    ROOTS=[MYDRIVE]
    sd=Path('/content/drive/Shareddrives')
    if sd.is_dir(): ROOTS.append(sd)

hint=os.environ.get('PHASE47_DRIVE_FOLDER_HINT','').strip()
if hint:
    hp=Path(hint).expanduser()
    if not hp.is_absolute(): hp=MYDRIVE/hp
    if not hp.is_dir(): raise FileNotFoundError(hp)
    ROOTS=[hp.resolve()]

WORK=Path(tempfile.mkdtemp(prefix='ichi2027_phase47_'))
SRC=WORK/'release'; SRC.mkdir()
STAGE=WORK/'outputs'; STAGE.mkdir()
OUT_BASE=Path(os.environ.get('PHASE47_OUTPUT_DIR',str(MYDRIVE/'ICHI2027'/'Phase47')))
OUT=OUT_BASE/dt.datetime.now(dt.timezone.utc).strftime('run_%Y%m%dT%H%M%S_%fZ')
print('Phase 47 output:',OUT)


Mounted at /content/drive
Phase 47 output: /content/drive/MyDrive/ICHI2027/Phase47/run_20261001T025353_969683Z


In [2]:
# CODE CELL 2/9 — Verify exact Phase 46 chain and exact Phase 41 challenge catalog

def name_matches(actual,stem,ext):
    return re.fullmatch(re.escape(stem)+r'(?:-\d+|\s*\(\d+\))?'+re.escape(ext),actual,re.I) is not None

def find_by_sha(stem,ext,digest,required=True):
    good=[]
    for root in ROOTS:
        for here,dirs,files in os.walk(root,topdown=True,followlinks=False):
            dirs[:]=[d for d in dirs if d not in {'.git','node_modules','.ipynb_checkpoints'} and not d.startswith('.Trash')]
            for fn in files:
                if name_matches(fn,stem,ext):
                    p=Path(here)/fn
                    try:
                        if sha256(p)==digest: good.append(p)
                    except OSError: pass
    if not good:
        if required: raise FileNotFoundError(f'No SHA-approved Drive copy found for {stem}{ext}')
        return None
    return sorted(good,key=str)[0]

P46=find_by_sha('ICHI2027_Phase46_SHAREABLE','.json',EXPECTED_P46_SHAREABLE)
L46=find_by_sha('ICHI2027_Phase46_TFL_V15_RELEASE_LOCK','.json',EXPECTED_P46_LOCK)
C41=find_by_sha('ICHI2027_Phase41_CHALLENGE_CATALOG','.csv',EXPECTED_P41_CHALLENGE)

p46=json.loads(P46.read_text('utf-8'))
l46=json.loads(L46.read_text('utf-8'))
with C41.open(newline='',encoding='utf-8-sig') as f:
    challenge41=list(csv.DictReader(f))

assert p46['phase']=='46_frozen_generic_tfl_v15_armC_acquisition_and_portability_preflight'
assert p46['frozen_commit_sha']==COMMIT
assert p46['frozen_archive_sha256']==EXPECTED_COMMIT_ARCHIVE_SHA
assert p46['frozen_original_generic_tfl_armC_executed'] is False
assert l46['commit_sha']==COMMIT
assert l46['archive_sha256']==EXPECTED_COMMIT_ARCHIVE_SHA
assert l46['comparator_rule']=='Use frozen generic v15 without neuroimaging-specific rule modifications'
assert len(challenge41)==8

print('PASS: Phase 46 comparator lock and Phase 41 E0–E7 catalog verified.')
print('Frozen commit:',COMMIT)
print('Challenge scenarios:',len(challenge41))


PASS: Phase 46 comparator lock and Phase 41 E0–E7 catalog verified.
Frozen commit: a4b7265adabf096e84b957de068e24a5ab853425
Challenge scenarios: 8


In [3]:
# CODE CELL 3/9 — Download and verify the exact frozen commit archive
import requests

session=requests.Session()
session.headers.update({'User-Agent':'ICHI2027-neuroimaging-fhir-omop-fidelity-phase47'})

archive=WORK/'tfl_v15_commit.zip'
url=f'https://api.github.com/repos/{REPO}/zipball/{COMMIT}'
r=session.get(url,timeout=120,headers={'Accept':'application/vnd.github+json'})
r.raise_for_status()
archive.write_bytes(r.content)

assert sha256(archive)==EXPECTED_COMMIT_ARCHIVE_SHA,(
    'Frozen commit archive hash changed or the wrong archive was returned.'
)

with zipfile.ZipFile(archive) as z:
    assert z.testzip() is None
    z.extractall(SRC)

tops=[p for p in SRC.iterdir() if p.is_dir()]
assert len(tops)==1
ROOT=tops[0]
RUNNER=ROOT/RUNNER_NAME
assert RUNNER.is_file()

print('PASS: exact Phase 46 archive re-downloaded and verified.')
print('Runner SHA-256:',sha256(RUNNER))


PASS: exact Phase 46 archive re-downloaded and verified.
Runner SHA-256: 5416d658e18d9d4f3dfb229996d685bd3dbeeb1d5d9ac910443b8527e9f3ff19


In [4]:
# CODE CELL 4/9 — Parse the authoritative v15 runner without modifying it

runner_nb=json.loads(RUNNER.read_text('utf-8'))
code_cells=[]
markdown_cells=[]
for idx,c in enumerate(runner_nb.get('cells',[])):
    src=''.join(c.get('source',[])) if isinstance(c.get('source',[]),list) else str(c.get('source',''))
    if c.get('cell_type')=='code':
        code_cells.append((idx,src))
    elif c.get('cell_type')=='markdown':
        markdown_cells.append((idx,src))

assert code_cells
runner_code='\n\n'.join(src for _,src in code_cells)
runner_text=(runner_code+'\n'+'\n'.join(src for _,src in markdown_cells))
runner_low=runner_text.lower()

cell_inventory=[]
for idx,src in code_cells:
    cell_inventory.append({
      'cell_index':idx,
      'source_sha256':hashlib.sha256(src.encode('utf-8')).hexdigest(),
      'source_chars':len(src),
      'contains_drive_mount':'drive.mount' in src,
      'contains_input_call':bool(re.search(r'\binput\s*\(',src)),
      'contains_files_upload':'files.upload' in src,
      'contains_shell_magic':any(line.lstrip().startswith(('!','%')) for line in src.splitlines())
    })

warning_evidence=[]
for w in FROZEN_WARNINGS:
    warning_evidence.append({
      'warning_code':w,
      'present_in_authoritative_runner':w in runner_text,
      'occurrences':runner_text.count(w)
    })

fate_evidence=[]
for s in FROZEN_FATES:
    fate_evidence.append({
      'fidelity_status':s,
      'present_in_authoritative_runner':s in runner_text,
      'occurrences':runner_text.count(s)
    })

print('Runner code cells:',len(code_cells))
print('Frozen warnings present:',sum(r['present_in_authoritative_runner'] for r in warning_evidence),'/',len(FROZEN_WARNINGS))
print('Frozen fates present:',sum(r['present_in_authoritative_runner'] for r in fate_evidence),'/',len(FROZEN_FATES))


Runner code cells: 17
Frozen warnings present: 6 / 7
Frozen fates present: 4 / 7


In [5]:
# CODE CELL 5/9 — Dependency, interactivity, and external-asset audit

stdlib=set(getattr(sys,'stdlib_module_names',set()))
imports=set()
syntax_failures=[]

for idx,src in code_cells:
    cleaned=[]
    for line in src.splitlines():
        if line.lstrip().startswith(('!','%')):
            continue
        cleaned.append(line)
    py='\n'.join(cleaned)
    try:
        tree=ast.parse(py)
        for node in ast.walk(tree):
            if isinstance(node,ast.Import):
                for a in node.names: imports.add(a.name.split('.')[0])
            elif isinstance(node,ast.ImportFrom) and node.module:
                imports.add(node.module.split('.')[0])
    except SyntaxError as e:
        syntax_failures.append({'cell_index':idx,'error':str(e)})

dep_rows=[]
for mod in sorted(imports):
    if mod in stdlib:
        status='STDLIB'
    else:
        try:
            status='AVAILABLE' if importlib.util.find_spec(mod) is not None else 'MISSING'
        except Exception:
            status='UNKNOWN'
    dep_rows.append({'module':mod,'status_in_current_environment':status})

interactive_tokens={
 'drive.mount':'drive.mount' in runner_code,
 'input_call':bool(re.search(r'\binput\s*\(',runner_code)),
 'files_upload':'files.upload' in runner_code,
 'getpass':'getpass(' in runner_code,
}
interactive_blocked=any(interactive_tokens.values())

# Extract literal paths/filenames referenced by the runner.
literal_strings=[]
for idx,src in code_cells:
    cleaned='\n'.join(line for line in src.splitlines() if not line.lstrip().startswith(('!','%')))
    try:
        tree=ast.parse(cleaned)
    except SyntaxError:
        continue
    for node in ast.walk(tree):
        if isinstance(node,ast.Constant) and isinstance(node.value,str):
            s=node.value
            if any(x in s.lower() for x in ['.csv','.json','.zip','.db','.sqlite','.ndjson','.parquet','.gz','/content/drive']):
                literal_strings.append(s)

asset_rows=[]
for s in sorted(set(literal_strings)):
    # We do not infer a local path for arbitrary literals; this is just dependency evidence.
    asset_rows.append({
      'literal_reference':s[:500],
      'looks_like_drive_path':'/content/drive' in s,
      'looks_like_data_asset':bool(re.search(r'\.(csv|json|zip|db|sqlite|ndjson|parquet|gz)\b',s,re.I))
    })

print('Parsed imports:',len(dep_rows))
print('Missing modules:',sum(r['status_in_current_environment']=='MISSING' for r in dep_rows))
print('Interactive/external mount tokens:',interactive_tokens)
print('Literal data/path references:',len(asset_rows))


Parsed imports: 14
Missing modules: 0
Interactive/external mount tokens: {'drive.mount': True, 'input_call': False, 'files_upload': False, 'getpass': False}
Literal data/path references: 22


In [6]:
# CODE CELL 6/9 — Audit actual frozen rule/resource scope

FHIR_TYPES=[
 'Patient','Encounter','Condition','Observation','MedicationRequest','Medication',
 'Procedure','Immunization','ImagingStudy','Device','Provenance'
]
resource_scope=[]
for rt in FHIR_TYPES:
    resource_scope.append({
      'resource_type':rt,
      'runner_text_occurrences':runner_text.count(rt),
      'explicitly_present_in_runner':rt in runner_text
    })

domain_terms=[
 'gender','duplicate','encounter','condition','coding','medication','reference',
 'ImagingStudy','IMAGE_OCCURRENCE','IMAGE_FEATURE','laterality','hippocamp',
 'entorhin','UCUM','mm3','timepoint','baseline','12-month','TRACE_STATE',
 'REVIEW_REQUIRED','algorithm_version'
]
term_scope=[]
for term in domain_terms:
    term_scope.append({
      'term':term,
      'occurrences_case_insensitive':runner_low.count(term.lower()),
      'present':term.lower() in runner_low
    })

# Function inventory: useful to determine whether the comparator exposes generic reusable callables.
function_rows=[]
for idx,src in code_cells:
    cleaned='\n'.join(line for line in src.splitlines() if not line.lstrip().startswith(('!','%')))
    try:
        tree=ast.parse(cleaned)
    except SyntaxError:
        continue
    for node in tree.body:
        if isinstance(node,(ast.FunctionDef,ast.AsyncFunctionDef)):
            function_rows.append({
              'cell_index':idx,
              'function_name':node.name,
              'arguments':'|'.join(a.arg for a in node.args.args),
              'source_sha256':hashlib.sha256(ast.get_source_segment(cleaned,node).encode('utf-8')).hexdigest()
                if ast.get_source_segment(cleaned,node) else ''
            })

generic_candidate_functions=[
    r for r in function_rows
    if any(t in r['function_name'].lower() for t in ['audit','fidel','warning','lineage','trace','map'])
]

print('Top-level function definitions:',len(function_rows))
print('Potential generic TFL callables:',len(generic_candidate_functions))
print('ImagingStudy explicit in runner:',any(r['resource_type']=='ImagingStudy' and r['explicitly_present_in_runner'] for r in resource_scope))


Top-level function definitions: 8
Potential generic TFL callables: 5
ImagingStudy explicit in runner: False


In [7]:
# CODE CELL 7/9 — Documentation-derived E0–E7 applicability preregistration

# This is frozen BEFORE any future Arm C neuroimaging run.
# It describes what the published/documented generic warning vocabulary is designed to detect.
# It is NOT an execution result and does not impute warnings to cases.

applicability={
 'E0':{
   'fault_family':'clean synthetic reference',
   'generic_v15_expected_scope':'BASELINE_PROFILE_ONLY',
   'applicable_warnings':'',
   'expected_generic_limitation':'No imaging-specific preservation assertion'
 },
 'E1':{
   'fault_family':'laterality swapped or missing',
   'generic_v15_expected_scope':'OUT_OF_SCOPE_IMAGING_SEMANTICS',
   'applicable_warnings':'',
   'expected_generic_limitation':'Frozen warning vocabulary has no laterality/anatomy rule'
 },
 'E2':{
   'fault_family':'mm3/cm3 normalization or missing unit',
   'generic_v15_expected_scope':'OUT_OF_SCOPE_IMAGING_UNIT_SEMANTICS',
   'applicable_warnings':'',
   'expected_generic_limitation':'Frozen warning vocabulary has no UCUM or imaging-value/unit preservation rule'
 },
 'E3':{
   'fault_family':'baseline/12-month source or visit collision',
   'generic_v15_expected_scope':'PARTIAL_ONLY_IF_GENERIC_IDENTITY_OR_REFERENCE_FAILURE_MANIFESTS',
   'applicable_warnings':'W_DUPLICATE_SOURCE_ID|W_TRACEABILITY_LOSS|W_UNRESOLVED_REFERENCE',
   'expected_generic_limitation':'Generic identity/reference warnings do not validate neuroimaging longitudinal timepoint semantics'
 },
 'E4':{
   'fault_family':'segmentation/extraction model version missing or changed',
   'generic_v15_expected_scope':'OUT_OF_SCOPE_MODEL_IDENTITY',
   'applicable_warnings':'',
   'expected_generic_limitation':'Frozen warning vocabulary has no neuroimaging extraction-model identity rule'
 },
 'E5':{
   'fault_family':'CAUTION/REVIEW_REQUIRED or reliability state stripped',
   'generic_v15_expected_scope':'OUT_OF_SCOPE_RELIABILITY_STATE',
   'applicable_warnings':'',
   'expected_generic_limitation':'Frozen warning vocabulary does not define TRACE/reliability-state preservation'
 },
 'E6':{
   'fault_family':'duplicate resources/references or unresolved source',
   'generic_v15_expected_scope':'GENERIC_IDENTITY_REFERENCE_SCOPE',
   'applicable_warnings':'W_DUPLICATE_SOURCE_ID|W_TRACEABILITY_LOSS|W_UNRESOLVED_REFERENCE|W_UNMAPPED_RESOURCE',
   'expected_generic_limitation':'Can address generic identity/reference conditions but not imaging-specific cardinality meaning'
 },
 'E7':{
   'fault_family':'combined/holdout faults',
   'generic_v15_expected_scope':'COMPONENT_DEPENDENT',
   'applicable_warnings':'W_DUPLICATE_SOURCE_ID|W_TRACEABILITY_LOSS|W_UNRESOLVED_REFERENCE|W_UNMAPPED_RESOURCE',
   'expected_generic_limitation':'Only generic components are expected to be in scope; imaging-specific components remain out of scope'
 }
}

challenge_rows=[]
for raw in challenge41:
    scenario=raw.get('scenario') or raw.get('scenario_id') or raw.get('challenge') or raw.get('challenge_id')
    if not scenario:
        # fall back to first column value only when it is E0–E7
        for v in raw.values():
            if str(v).strip() in applicability:
                scenario=str(v).strip(); break
    assert scenario in applicability,(scenario,raw)
    a=applicability[scenario]
    challenge_rows.append({
      'scenario':scenario,
      'phase41_catalog_row_sha256':hashlib.sha256(
          json.dumps(raw,sort_keys=True,separators=(',',':')).encode('utf-8')
      ).hexdigest(),
      **a,
      'classification_kind':'PREREGISTERED_DOCUMENTATION_DERIVED_EXPECTATION_NOT_ARM_C_RESULT'
    })

assert {r['scenario'] for r in challenge_rows}==set(applicability)
print('PASS: E0–E7 generic-v15 applicability expectation frozen for future comparison.')
for r in challenge_rows:
    print(r['scenario'],r['generic_v15_expected_scope'])


PASS: E0–E7 generic-v15 applicability expectation frozen for future comparison.
E0 BASELINE_PROFILE_ONLY
E1 OUT_OF_SCOPE_IMAGING_SEMANTICS
E2 OUT_OF_SCOPE_IMAGING_UNIT_SEMANTICS
E3 PARTIAL_ONLY_IF_GENERIC_IDENTITY_OR_REFERENCE_FAILURE_MANIFESTS
E4 OUT_OF_SCOPE_MODEL_IDENTITY
E5 OUT_OF_SCOPE_RELIABILITY_STATE
E6 GENERIC_IDENTITY_REFERENCE_SCOPE
E7 COMPONENT_DEPENDENT


In [8]:
# CODE CELL 8/9 — Attempt unchanged v15 execution only when safe/non-interactive

execution_status='NOT_ATTEMPTED'
execution_returncode=None
execution_tail=''
executed_notebook_sha256=''
execution_seconds=None

missing_modules=[r['module'] for r in dep_rows if r['status_in_current_environment']=='MISSING']

# Do not patch the runner. If it requires interactive Drive mounting or missing modules,
# record the blocker rather than editing the frozen comparator.
if interactive_blocked:
    execution_status='BLOCKED_INTERACTIVE_OR_DRIVE_DEPENDENCY'
elif missing_modules:
    execution_status='BLOCKED_MISSING_RUNTIME_DEPENDENCIES'
else:
    try:
        import nbformat
        from nbclient import NotebookClient
        from nbclient.exceptions import CellExecutionError

        nb=nbformat.read(str(RUNNER),as_version=4)
        t0=time.time()
        client=NotebookClient(
            nb,
            timeout=180,
            kernel_name='python3',
            resources={'metadata':{'path':str(ROOT)}},
            allow_errors=False
        )
        try:
            client.execute()
            execution_status='PASS_UNMODIFIED_NATIVE_V15_RUNNER'
        except CellExecutionError as e:
            execution_status='FAILED_UNMODIFIED_NATIVE_V15_RUNNER'
            execution_tail=str(e)[-6000:]
        except Exception as e:
            execution_status='FAILED_UNMODIFIED_NATIVE_V15_RUNNER'
            execution_tail=repr(e)[-6000:]
        execution_seconds=round(time.time()-t0,3)
        executed=WORK/'executed_v15.ipynb'
        nbformat.write(nb,str(executed))
        executed_notebook_sha256=sha256(executed)
    except Exception as e:
        execution_status='EXECUTION_FRAMEWORK_ERROR'
        execution_tail=repr(e)[-6000:]

# This still does not count as neuroimaging Arm C execution.
armc_neuroimaging_executed=False

if execution_status=='PASS_UNMODIFIED_NATIVE_V15_RUNNER':
    next_status='NATIVE_V15_EXECUTABLE_BUT_NEUROIMAGING_ARM_C_NOT_RUN'
elif execution_status.startswith('BLOCKED_'):
    next_status='NATIVE_V15_EXECUTION_BLOCKER_DOCUMENTED'
else:
    next_status='NATIVE_V15_RUNNER_EXECUTION_FAILED_OR_NOT_REPRODUCED'

print('Unmodified native v15 execution:',execution_status)
print('Neuroimaging Arm C executed:',armc_neuroimaging_executed)
print('Comparator status:',next_status)


Unmodified native v15 execution: BLOCKED_INTERACTIVE_OR_DRIVE_DEPENDENCY
Neuroimaging Arm C executed: False
Comparator status: NATIVE_V15_EXECUTION_BLOCKER_DOCUMENTED


In [9]:
# CODE CELL 9/9 — Save public Phase 47 evidence

def save(name,rows,fields=None):
    p=STAGE/name
    write_csv(p,rows,fields)
    return p

artifacts=[]
artifacts.append(save('ICHI2027_Phase47_V15_RUNNER_CELL_INVENTORY.csv',cell_inventory))
artifacts.append(save('ICHI2027_Phase47_DEPENDENCY_AUDIT.csv',dep_rows if dep_rows else [
 {'module':'','status_in_current_environment':'NO_IMPORTS_PARSED'}
]))
artifacts.append(save('ICHI2027_Phase47_EXTERNAL_ASSET_REFERENCES.csv',asset_rows if asset_rows else [
 {'literal_reference':'','looks_like_drive_path':False,'looks_like_data_asset':False}
]))
artifacts.append(save('ICHI2027_Phase47_FROZEN_WARNING_EVIDENCE.csv',warning_evidence))
artifacts.append(save('ICHI2027_Phase47_FROZEN_FATE_EVIDENCE.csv',fate_evidence))
artifacts.append(save('ICHI2027_Phase47_RESOURCE_SCOPE.csv',resource_scope))
artifacts.append(save('ICHI2027_Phase47_TERM_SCOPE.csv',term_scope))
artifacts.append(save('ICHI2027_Phase47_FUNCTION_INVENTORY.csv',function_rows if function_rows else [
 {'cell_index':'','function_name':'','arguments':'','source_sha256':''}
]))
artifacts.append(save('ICHI2027_Phase47_E0_E7_GENERIC_V15_APPLICABILITY.csv',challenge_rows))

execution_rows=[{
 'frozen_commit':COMMIT,
 'runner_name':RUNNER_NAME,
 'runner_sha256':sha256(RUNNER),
 'interactive_blocked':interactive_blocked,
 'missing_module_count':len(missing_modules),
 'execution_status':execution_status,
 'execution_returncode':execution_returncode if execution_returncode is not None else '',
 'execution_seconds':execution_seconds if execution_seconds is not None else '',
 'executed_notebook_sha256':executed_notebook_sha256,
 'neuroimaging_armC_executed':False,
 'comparator_status':next_status,
 'error_tail_public':execution_tail.replace('\n',' ')[:2000]
}]
artifacts.append(save('ICHI2027_Phase47_NATIVE_V15_EXECUTION_AUDIT.csv',execution_rows))

readme=STAGE/'ICHI2027_Phase47_README.txt'
readme.write_text(
 'Phase 47 audits the authoritative frozen v15 integration notebook without modifying its method code.\n'
 'The E0–E7 applicability table is preregistered from the documented frozen generic warning vocabulary and is not an Arm C result.\n'
 'A successful native v15 runner execution would establish reproducibility in its own frozen workflow only; it would not establish neuroimaging Arm C execution.\n'
 'No MRI-specific rules are added to v15.\n'
 'The neuroimaging comparator remains unexecuted until the frozen generic method is actually applied through a documented unmodified rule path to the study evidence.\n',
 encoding='utf-8'
)
artifacts.append(readme)

summary={
 'project':PROJECT,
 'phase':PHASE,
 'run_utc':dt.datetime.now(dt.timezone.utc).isoformat(),
 'input_phase46_shareable_sha256':sha256(P46),
 'input_phase46_lock_sha256':sha256(L46),
 'input_phase41_challenge_catalog_sha256':sha256(C41),
 'frozen_repository':REPO,
 'frozen_tag':TAG,
 'frozen_commit':COMMIT,
 'verified_commit_archive_sha256':sha256(archive),
 'authoritative_runner':RUNNER_NAME,
 'authoritative_runner_sha256':sha256(RUNNER),
 'runner_code_cells':len(code_cells),
 'frozen_warning_codes_evidenced':sum(r['present_in_authoritative_runner'] for r in warning_evidence),
 'frozen_warning_codes_total':len(FROZEN_WARNINGS),
 'frozen_fate_states_evidenced':sum(r['present_in_authoritative_runner'] for r in fate_evidence),
 'frozen_fate_states_total':len(FROZEN_FATES),
 'parsed_import_modules':len(dep_rows),
 'missing_runtime_modules':len(missing_modules),
 'interactive_or_drive_dependency_detected':interactive_blocked,
 'top_level_functions_found':len(function_rows),
 'potential_generic_tfl_callables_found':len(generic_candidate_functions),
 'native_v15_execution_status':execution_status,
 'native_v15_execution_seconds':execution_seconds,
 'e0_e7_applicability_rows':len(challenge_rows),
 'applicability_matrix_kind':'PREREGISTERED_DOCUMENTATION_DERIVED_EXPECTATION_NOT_ARM_C_RESULT',
 'frozen_original_generic_tfl_armC_executed':False,
 'armC_neuroimaging_execution_status':next_status,
 'armA_evidence_supported_gates_carried_forward':p46['armA_evidence_supported_gates_carried_forward'],
 'armA_remaining_gates_carried_forward':p46['armA_remaining_gates_carried_forward'],
 'official_micdm_load_executed':False,
 'official_omop_etl_executed':False,
 'real_authorized_mri_processed':False,
 'full_A_B_C_D_evaluation':False,
 'artifact_sha256':{p.name:sha256(p) for p in artifacts},
 'warning':'Native frozen-runner reproducibility and documentation-derived applicability are not substitutes for actual Arm C execution on neuroimaging evidence.'
}

summary_path=STAGE/'ICHI2027_Phase47_SHAREABLE.json'
summary_path.write_text(json.dumps(summary,sort_keys=True,indent=2)+'\n',encoding='utf-8')
artifacts.append(summary_path)

zip_path=STAGE/'ICHI2027_Phase47_PUBLIC_OUTPUTS.zip'
with zipfile.ZipFile(zip_path,'w',compression=zipfile.ZIP_DEFLATED) as z:
    for p in artifacts:z.write(p,p.name)

OUT.mkdir(parents=True,exist_ok=False)
for p in artifacts+[zip_path]:
    dest=OUT/p.name
    for attempt in range(3):
        try:
            shutil.copyfile(p,dest)
            assert sha256(dest)==sha256(p)
            break
        except Exception as e:
            if attempt==2: raise RuntimeError('Drive save/readback verification failed for '+p.name) from e
            time.sleep(1+attempt)

print('SUCCESS: Phase 47 frozen-v15 runner audit completed.')
print('Native v15 execution:',execution_status)
print('Neuroimaging Arm C executed: False')
print('Shareable:',OUT/summary_path.name)
print('Public ZIP:',OUT/zip_path.name)


SUCCESS: Phase 47 frozen-v15 runner audit completed.
Native v15 execution: BLOCKED_INTERACTIVE_OR_DRIVE_DEPENDENCY
Neuroimaging Arm C executed: False
Shareable: /content/drive/MyDrive/ICHI2027/Phase47/run_20261001T025353_969683Z/ICHI2027_Phase47_SHAREABLE.json
Public ZIP: /content/drive/MyDrive/ICHI2027/Phase47/run_20261001T025353_969683Z/ICHI2027_Phase47_PUBLIC_OUTPUTS.zip


## Decision after Phase 47

The next step depends on the evidence produced here:

- If the unmodified v15 runner executes and exposes a reusable generic callable that can accept new resource evidence without altering rule definitions, the next phase can build a **thin input adapter** and test that exact frozen logic on the E0–E7 study fixtures.
- If the frozen runner is hard-coded to its original clinical-resource workflow, the correct comparison is to document that portability ceiling and keep the generic v15 baseline separate rather than rewriting it into an imaging-aware comparator.
- Arm A remains governed by the independent source/vocabulary gates from Phase 45.
